# Evaluation Notebook (02)

Dieses Notebook sammelt 20+ kuratierte Queries und berechnet Kernmetriken für den Mixing Forum Analyzer. Die Ergebnisse fließen in das README und das interne Metriken-Dashboard.

## 1. Setup & Imports

In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from evaluation.metrics import (
    cosine_similarity_distribution,
    inference_latency_summary,
    mean_reciprocal_rank,
    top_k_accuracy,
)
from preset_advisor.search import SemanticSearchEngine

CORPUS_PATH = Path('data/sample_corpus.json')
corpus = json.loads(CORPUS_PATH.read_text(encoding='utf-8'))
doc_ids = [f'doc_{idx}' for idx, _ in enumerate(corpus)]
doc_lookup = dict(zip(doc_ids, corpus))

len(corpus), list(doc_lookup.items())[:3]


## 2. Manuell kuratierte Queries

In [ ]:
EVAL_QUERIES = [
  {
    "query": "Snare klingt zu harsch und laut",
    "relevant": [
      "doc_0",
      "doc_13"
    ]
  },
  {
    "query": "Kick braucht mehr Punch und Attack",
    "relevant": [
      "doc_1",
      "doc_10",
      "doc_30"
    ]
  },
  {
    "query": "Vocals nasal 800 Hz reduzieren",
    "relevant": [
      "doc_2"
    ]
  },
  {
    "query": "Bass maskiert Kick, brauche Sidechain",
    "relevant": [
      "doc_3",
      "doc_25"
    ]
  },
  {
    "query": "S-Laute scharf, welcher De-Esser?",
    "relevant": [
      "doc_4",
      "doc_9"
    ]
  },
  {
    "query": "Akustikgitarre klingt blechern bei 3khz",
    "relevant": [
      "doc_5"
    ]
  },
  {
    "query": "Hi-Hats zischeln zu stark",
    "relevant": [
      "doc_6",
      "doc_24"
    ]
  },
  {
    "query": "Synth Pad verdeckt Vocals, M/S EQ?",
    "relevant": [
      "doc_7",
      "doc_26"
    ]
  },
  {
    "query": "Snare verliert Sustain, brauche Attack",
    "relevant": [
      "doc_8"
    ]
  },
  {
    "query": "Kick klingt porös, will mehr Sub bei 60hz",
    "relevant": [
      "doc_10"
    ]
  },
  {
    "query": "Bassline mulmt bei 200hz",
    "relevant": [
      "doc_11",
      "doc_22"
    ]
  },
  {
    "query": "Lead Vocals zu leise, parallel komprimieren?",
    "relevant": [
      "doc_12",
      "doc_29"
    ]
  },
  {
    "query": "Backing Vocals zu breit, Mono Probleme",
    "relevant": [
      "doc_14"
    ]
  },
  {
    "query": "Toms resonieren bei 120hz",
    "relevant": [
      "doc_15"
    ]
  },
  {
    "query": "Ride Becken zu laut",
    "relevant": [
      "doc_16"
    ]
  },
  {
    "query": "Masterbus pumpt, Release zu kurz",
    "relevant": [
      "doc_17"
    ]
  },
  {
    "query": "Gitarrenwand frisst Präsenz, Vocals gehen unter",
    "relevant": [
      "doc_18"
    ]
  },
  {
    "query": "808 Sustain zu lang, wie kürzen?",
    "relevant": [
      "doc_19"
    ]
  },
  {
    "query": "Piano hat harte Transienten, Soft Clipper?",
    "relevant": [
      "doc_20"
    ]
  },
  {
    "query": "Raummikros haben Phase Probleme",
    "relevant": [
      "doc_21"
    ]
  },
  {
    "query": "Snare braucht mehr Body um 200hz",
    "relevant": [
      "doc_22"
    ]
  },
  {
    "query": "Delay auf Vocals verschmiert, Highcut nötig?",
    "relevant": [
      "doc_23"
    ]
  },
  {
    "query": "Hi-Hats brauchen mehr Shimmer",
    "relevant": [
      "doc_24",
      "doc_6"
    ]
  },
  {
    "query": "Kick und Bass sind phasig",
    "relevant": [
      "doc_25"
    ]
  },
  {
    "query": "Synth maskiert Vocals trotz EQ",
    "relevant": [
      "doc_26",
      "doc_7"
    ]
  },
  {
    "query": "Drumbus wirkt zu schmal, mehr Stereo",
    "relevant": [
      "doc_27"
    ]
  },
  {
    "query": "Gate auf Snare arbeitet nicht sauber",
    "relevant": [
      "doc_28"
    ]
  },
  {
    "query": "Vox zu dynamisch, suche Leveling",
    "relevant": [
      "doc_29",
      "doc_12"
    ]
  },
  {
    "query": "Kick zu laut im Mix",
    "relevant": [
      "doc_30",
      "doc_1"
    ]
  }
]
len(EVAL_QUERIES)

## 3. Suchengine laden und Ergebnisse erzeugen

In [ ]:
engine = SemanticSearchEngine(corpus)
query_texts = [entry['query'] for entry in EVAL_QUERIES]
relevant_lists = [entry['relevant'] for entry in EVAL_QUERIES]

results = engine.batch_query(query_texts, top_k=5)
ranked_doc_ids = [[res.doc_id for res in query_results] for query_results in results]
score_matrix = np.array([[res.score for res in query_results] for query_results in results])
ranked_doc_ids[:2]


## 4. Retrieval-Metriken

In [ ]:
mrr = mean_reciprocal_rank(ranked_doc_ids, relevant_lists)
topk = top_k_accuracy(ranked_doc_ids, relevant_lists, ks=(1, 3, 5))
sim_summary = cosine_similarity_distribution(score_matrix.flatten())

metric_table = pd.DataFrame([
    {"Metrik": "MRR", "Wert": round(mrr, 3), "Benchmark": "> 0.70"},
    {"Metrik": "Top-1-Accuracy", "Wert": round(topk[1], 3), "Benchmark": "> 0.60"},
    {"Metrik": "Top-3-Accuracy", "Wert": round(topk[3], 3), "Benchmark": "> 0.80"},
    {"Metrik": "Top-5-Accuracy", "Wert": round(topk[5], 3), "Benchmark": "> 0.90"},
])
metric_table


In [ ]:
sim_summary.as_dict()

## 5. Inferenz-Latenz

In [ ]:
latencies = []
for _ in range(3):
    for query in query_texts:
        start = time.perf_counter()
        engine.query(query, top_k=5)
        latencies.append((time.perf_counter() - start) * 1000)

lat_summary = inference_latency_summary(latencies)
lat_summary.as_dict()


## 6. Confusion-Analyse

In [ ]:
analysis_rows = []
for query, relevant, query_results in zip(query_texts, relevant_lists, results):
    top_hit = query_results[0] if query_results else None
    predicted_id = top_hit.doc_id if top_hit else None
    predicted_score = top_hit.score if top_hit else None
    relevant_set = set(relevant)
    hits_in_top_k = [res.doc_id for res in query_results if res.doc_id in relevant_set]
    missed = list(relevant_set.difference(hits_in_top_k))
    analysis_rows.append(
        {
            "query": query,
            "top_hit": predicted_id,
            "score": round(predicted_score, 3) if predicted_score is not None else None,
            "is_hit": bool(predicted_id and predicted_id in relevant_set),
            "false_negative": bool(missed),
            "missed_relevant": ", ".join(missed) if missed else "",
        }
    )

confusion_df = pd.DataFrame(analysis_rows)
confusion_df.head(10)


## 7. Visualisierungen

In [ ]:
try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(10, 4))
    plt.hist(score_matrix.flatten(), bins=15, range=(0, 1), color="#5078F2", alpha=0.7)
    plt.title("Cosine Similarity Distribution (Top-5 Scores)")
    plt.xlabel("Score")
    plt.ylabel("Häufigkeit")
    plt.grid(alpha=0.2)
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.boxplot(score_matrix.flatten(), vert=False)
    plt.title("Score Spread")
    plt.xlabel("Cosine Similarity")
    plt.grid(alpha=0.2)
    plt.show()
except ModuleNotFoundError:
    print("Matplotlib nicht installiert - Diagramme übersprungen.")


## 8. Export für README

In [ ]:
output_path = Path('docs') / 'metrics_for_readme.md'
output_path.parent.mkdir(exist_ok=True)
with output_path.open('w', encoding='utf-8') as fh:
    fh.write(metric_table.to_markdown(index=False))

output_path
